# Chapitre 4 : séparer les données des instructions

- [Cours](#cours)
- [Exercices](#exercices)
- [Espace de test](#espace-de-test)

## Configuration

Exécutez la cellule de configuration suivante pour charger votre clé API et définir la fonction utilitaire `get_completion`.

In [ ]:
!pip install anthropic

# Importer la bibliothèque standard de Python pour les expressions régulières
import re
import anthropic

# Récupérer les variables API_KEY et MODEL_NAME depuis le store IPython
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

---

## Cours

Souvent, on ne veut pas écrire des prompts complets, mais plutôt des **modèles de prompt (templates) que l'on complète ensuite avec des données d'entrée avant de les envoyer à Claude**. C'est utile lorsque Claude doit faire la même chose à chaque fois, mais avec des données différentes.

Heureusement, c'est assez simple : il suffit de **séparer le squelette fixe du prompt de l'entrée variable, puis d'insérer cette entrée dans le prompt** avant d'envoyer le prompt complet à Claude.

Nous allons voir pas à pas comment écrire un modèle de prompt à variables, et comment y insérer l'entrée de l'utilisateur.

### Exemples

Dans ce premier exemple, nous demandons à Claude de jouer le rôle d'un générateur de cris d'animaux. Le prompt complet envoyé à Claude est simplement le `PROMPT_TEMPLATE` dans lequel on a inséré l'entrée (ici « Cow », vache). Lorsque nous affichons le prompt complet, le mot « Cow » remplace la variable `ANIMAL` grâce à une f-string.

**Note :** en pratique, le nom de la variable importe peu. Nous l'avons appelée `ANIMAL` ici, mais nous aurions pu l'appeler `CREATURE` ou `A` (il reste préférable de choisir des noms précis et parlants, pour que le modèle de prompt soit compréhensible même sans substitution). Veillez simplement à utiliser le même nom dans la f-string du modèle.

In [ ]:
# Contenu de la variable
ANIMAL = "Cow"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Pourquoi séparer et insérer les entrées de cette façon ? Parce que **les modèles de prompt simplifient les tâches répétitives**. Imaginez une structure de prompt dans laquelle des utilisateurs tiers soumettent du contenu (ici, l'animal dont ils veulent le cri). Ces utilisateurs n'ont pas besoin d'écrire ni même de voir le prompt complet : ils se contentent de remplir des variables.

Nous faisons ici la substitution avec des variables et des f-strings, mais la méthode format() fonctionne aussi.

**Note :** un modèle de prompt peut contenir autant de variables que nécessaire.

Lorsque vous introduisez des variables de ce type, il est essentiel de **faire en sorte que Claude sache où chaque variable commence et se termine** (par opposition aux instructions ou à la description de la tâche). Voyons un exemple sans séparation entre les instructions et la variable.

Pour un œil humain, on voit très bien où commence et finit la variable dans le modèle ci-dessous. Mais dans le prompt complet, après substitution, cette délimitation devient floue.

In [ ]:
# Contenu de la variable
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Ici, **Claude croit que « Yo Claude » fait partie de l'e-mail qu'il doit réécrire** ! On le voit parce que sa réécriture commence par « Dear Claude ». Pour un humain, surtout en regardant le modèle de prompt, on sait où l'e-mail commence et se termine, mais c'est beaucoup moins clair une fois la substitution faite.

Comment résoudre ce problème ? **En entourant l'entrée de balises XML** ! C'est ce que nous faisons ci-dessous, et comme vous le voyez, le « Dear Claude » disparaît de la réponse.

Les [balises XML](https://docs.anthropic.com/claude/docs/use-xml-tags) sont des balises entre chevrons comme `<tag></tag>`. Elles vont par paires : une balise ouvrante, comme `<tag>`, et une balise fermante marquée d'un `/`, comme `</tag>`. Elles encadrent un contenu, ainsi : `<tag>contenu</tag>`.

**Note :** Claude sait reconnaître et utiliser une grande variété de séparateurs et de délimiteurs, mais nous recommandons **d'utiliser précisément des balises XML** comme séparateurs, car Claude a été entraîné à les reconnaître comme mécanisme d'organisation du prompt. En dehors de l'appel de fonctions, **il n'existe pas de balises XML « magiques » sur lesquelles Claude aurait été entraîné et qui amélioreraient ses performances au maximum**. Nous avons volontairement rendu Claude très souple et personnalisable sur ce point.

In [ ]:
# Contenu de la variable
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Voyons un autre exemple de l'utilité des balises XML.

Dans le prompt suivant, **Claude se trompe sur ce qui relève de l'instruction et ce qui relève de l'entrée**. À cause de la mise en forme, il considère à tort que `Each is about an animal, like rabbits` fait partie de la liste, alors que l'utilisateur (celui qui remplit la variable `SENTENCES`) ne le voulait sans doute pas.

In [ ]:
# Contenu de la variable
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Pour corriger cela, il suffit **d'entourer de balises XML les phrases fournies par l'utilisateur**. Claude sait alors où commencent et finissent les données d'entrée, malgré le tiret trompeur devant `Each is about an animal, like rabbits.`

In [ ]:
# Contenu de la variable
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

**Note :** dans la version incorrecte du prompt « Each is about an animal », nous avons dû ajouter le tiret pour que Claude se trompe comme nous le voulions pour l'exemple. C'est une leçon importante : **les petits détails comptent** ! Il vaut toujours la peine de **relire vos prompts pour corriger fautes de frappe et erreurs de grammaire**. Claude est sensible aux motifs (à ses débuts, avant l'affinage, c'était un simple outil de prédiction de texte) : il fait plus d'erreurs quand vous en faites, il est plus pertinent quand vous l'êtes, plus fantaisiste quand vous l'êtes, et ainsi de suite.

Si vous voulez expérimenter avec les prompts du cours sans modifier le contenu ci-dessus, descendez tout en bas du notebook jusqu'à l'[**Espace de test**](#espace-de-test).

---

## Exercices
- [Exercice 4.1 - Sujet de haïku](#exercice-41---sujet-de-haïku)
- [Exercice 4.2 - Question sur les chiens, avec fautes](#exercice-42---question-sur-les-chiens-avec-fautes)
- [Exercice 4.3 - Question sur les chiens, partie 2](#exercice-43---question-sur-les-chiens-partie-2)

### Exercice 4.1 - Sujet de haïku
Modifiez le `PROMPT` pour en faire un modèle qui prend une variable `TOPIC` et produit un haïku sur ce sujet. Cet exercice vérifie simplement que vous avez compris la structure des modèles à variables avec les f-strings.

In [ ]:
# Contenu de la variable
TOPIC = "Pigs"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f""

# Obtenir la réponse de Claude
response = get_completion(PROMPT)

# Fonction de correction de l'exercice
def grade_exercise(text):
    return bool(re.search("pigs", text.lower()) and re.search("haiku", text.lower()))

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pour obtenir un indice, exécutez la cellule ci-dessous.

In [ ]:
from hints import exercise_4_1_hint; print(exercise_4_1_hint)

### Exercice 4.2 - Question sur les chiens, avec fautes
Corrigez le `PROMPT` en ajoutant des balises XML pour que Claude donne la bonne réponse.

Essayez de ne rien changer d'autre dans le prompt. La rédaction brouillonne et pleine de fautes est volontaire : elle montre comment Claude réagit à ce type d'erreurs.

In [ ]:
# Contenu de la variable
QUESTION = "ar cn brown?"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Obtenir la réponse de Claude
response = get_completion(PROMPT)

# Fonction de correction de l'exercice
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pour obtenir un indice, exécutez la cellule ci-dessous.

In [ ]:
from hints import exercise_4_2_hint; print(exercise_4_2_hint)

### Exercice 4.3 - Question sur les chiens, partie 2
Corrigez le `PROMPT` **SANS** ajouter de balises XML. Supprimez seulement un ou deux mots du prompt.

Comme pour les exercices précédents, essayez de ne rien changer d'autre. Vous verrez quel type de langage Claude sait analyser et comprendre.

In [ ]:
# Contenu de la variable
QUESTION = "ar cn brown?"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Obtenir la réponse de Claude
response = get_completion(PROMPT)

# Fonction de correction de l'exercice
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pour obtenir un indice, exécutez la cellule ci-dessous.

In [ ]:
from hints import exercise_4_3_hint; print(exercise_4_3_hint)

### Bravo !

Si vous avez résolu tous les exercices jusqu'ici, vous pouvez passer au chapitre suivant. Bon prompting !

---

## Espace de test

Cet espace vous permet d'expérimenter librement avec les exemples de prompts de la leçon et de les modifier pour voir l'effet sur les réponses de Claude.

In [ ]:
# Contenu de la variable
ANIMAL = "Cow"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Contenu de la variable
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Contenu de la variable
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Contenu de la variable
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Contenu de la variable
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))